# Used Car Price Intelligence & Prediction System

Predicts a used car's resale price from its details - brand, year, mileage, fuel type, and
more - using traditional ML (Random Forest, XGBoost) as the actual predictor, with an LLM
used for enrichment and comparison, not as the core engine.

| Day | Covers |
|---|---|
| 1 | Synthetic dataset generation, cleaning, EDA |
| 2 | LLM extracts structured fields from raw listing text |
| 3 | Linear Regression, Random Forest, XGBoost |
| 4 | Model comparison against baselines |
| 5 | Neural network + zero-shot LLM prediction, compared against trained ML |
| Final | Gradio dashboard |

In [ ]:
%pip install -q openai pandas numpy scikit-learn torch matplotlib seaborn gradio pydantic

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))

MODEL = "gpt-4.1-mini"
GROQ_MODEL = "openai/gpt-oss-120b"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

## dataset generation, cleaning, EDA

Real used-car pricing follows real patterns - cars depreciate with age and mileage, brand
tier matters, automatic/diesel carry a premium. This generates a dataset that follows those
patterns with realistic noise, plus deliberate messiness (missing values, duplicates,
outliers) so the cleaning step has genuine work to do.

In [ ]:
random.seed(42)
np.random.seed(42)

BRANDS = {
    # brand: (base_price, tier_multiplier)
    "Maruti Suzuki": 600000, "Hyundai": 700000, "Tata": 650000, "Honda": 800000,
    "Toyota": 900000, "Ford": 750000, "Volkswagen": 850000,
    "BMW": 2500000, "Mercedes-Benz": 2800000, "Audi": 2600000,
}
FUEL_TYPES = ["Petrol", "Diesel", "CNG", "Electric"]
TRANSMISSIONS = ["Manual", "Automatic"]
LOCATIONS = ["Mumbai", "Delhi", "Bangalore", "Pune", "Chennai", "Hyderabad"]

N_CARS = 3000
rows = []

for i in range(N_CARS):
    brand = random.choice(list(BRANDS.keys()))
    base_price = BRANDS[brand]
    year = random.randint(2010, 2024)
    age = 2025 - year
    km_driven = max(1000, int(np.random.normal(age * 12000, 8000)))
    fuel_type = random.choices(FUEL_TYPES, weights=[0.45, 0.35, 0.1, 0.1])[0]
    transmission = random.choices(TRANSMISSIONS, weights=[0.6, 0.4])[0]
    engine_cc = int(np.random.normal(1500 if base_price < 1000000 else 2500, 300))
    owner_count = random.choices([1, 2, 3, 4], weights=[0.5, 0.3, 0.15, 0.05])[0]
    location = random.choice(LOCATIONS)

    # Realistic price formula: depreciation by age and mileage, adjustments for fuel/transmission/owners
    price = base_price
    price *= (0.90 ** age)                                    # ~10% depreciation per year
    price *= max(0.4, 1 - (km_driven / 300000))               # mileage depreciation
    price *= 1.15 if fuel_type == "Electric" else 1.0
    price *= 0.90 if fuel_type == "CNG" else 1.0
    price *= 1.12 if transmission == "Automatic" else 1.0
    price *= (1 - 0.05 * (owner_count - 1))                   # each extra owner knocks off ~5%
    price *= np.random.normal(1.0, 0.08)                      # random market noise
    price = max(80000, price)

    rows.append({
        "brand": brand, "year": year, "km_driven": km_driven, "fuel_type": fuel_type,
        "transmission": transmission, "engine_cc": engine_cc, "owner_count": owner_count,
        "location": location, "price": round(price, -2)
    })

df = pd.DataFrame(rows)

# Inject realistic messiness - about 3% missing values in a couple of columns
missing_idx = df.sample(frac=0.03, random_state=1).index
df.loc[missing_idx, "engine_cc"] = np.nan
missing_idx2 = df.sample(frac=0.02, random_state=2).index
df.loc[missing_idx2, "km_driven"] = np.nan

# A handful of duplicate rows, as real scraped data often has
df = pd.concat([df, df.sample(20, random_state=3)], ignore_index=True)

# A few obvious data-entry-error outliers
outlier_idx = df.sample(5, random_state=4).index
df.loc[outlier_idx, "price"] = df.loc[outlier_idx, "price"] * 20

print(f"Generated {len(df)} rows (before cleaning)")
df.head()

In [ ]:
# Cleaning: drop duplicates, fill missing values sensibly, remove outliers

print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

print(f"Missing values:\n{df.isnull().sum()}")
df["engine_cc"] = df["engine_cc"].fillna(df.groupby("brand")["engine_cc"].transform("median"))
df["km_driven"] = df["km_driven"].fillna(df["km_driven"].median())

# Remove obvious price outliers using the IQR method
Q1, Q3 = df["price"].quantile(0.25), df["price"].quantile(0.75)
IQR = Q3 - Q1
before = len(df)
df = df[(df["price"] >= Q1 - 3 * IQR) & (df["price"] <= Q3 + 3 * IQR)].reset_index(drop=True)
print(f"Removed {before - len(df)} price outliers")

print(f"\nFinal clean dataset: {len(df)} rows")
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(df["year"], df["price"], alpha=0.3, s=10)
axes[0, 0].set_title("Price vs Year")
axes[0, 0].set_xlabel("Year")
axes[0, 0].set_ylabel("Price")

axes[0, 1].scatter(df["km_driven"], df["price"], alpha=0.3, s=10, color="orange")
axes[0, 1].set_title("Price vs Kilometers Driven")
axes[0, 1].set_xlabel("Km driven")

sns.boxplot(data=df, x="brand", y="price", ax=axes[1, 0])
axes[1, 0].set_title("Price by Brand")
axes[1, 0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df, x="fuel_type", y="price", ax=axes[1, 1])
axes[1, 1].set_title("Price by Fuel Type")

plt.tight_layout()
plt.show()

## LLM extracts structured fields from raw listing text

Real listings are messy free text, not clean columns. An LLM reads a raw listing description
and extracts the structured fields your ML models actually need - a genuinely useful
enrichment step, not just a demo.

In [ ]:
class ExtractedListing(BaseModel):
    brand: str
    year: int
    km_driven: int
    fuel_type: str
    transmission: str
    owner_count: int
    location: str


def extract_listing_info(raw_text, client=openai_client, model=MODEL):
    response = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": "Extract structured car listing details from this "
                                           "free-text description. Infer reasonable values for "
                                           "anything not explicitly stated."},
            {"role": "user", "content": raw_text}
        ],
        response_format=ExtractedListing,
    )
    return ExtractedListing.model_validate_json(response.choices[0].message.content)


raw_listing = """2018 Honda City, well maintained, 45000 kms driven, diesel, manual
transmission, single owner, located in Mumbai. Selling due to relocation, no accidents,
all service records available."""

extracted = extract_listing_info(raw_listing)
print(extracted)

## traditional ML models

Linear Regression as the baseline, then two ensemble models known for strong performance on
structured tabular data.

In [ ]:
FEATURE_COLUMNS = ["brand", "year", "km_driven", "fuel_type", "transmission",
                   "engine_cc", "owner_count", "location"]
CATEGORICAL_COLUMNS = ["brand", "fuel_type", "transmission", "location"]
NUMERIC_COLUMNS = ["year", "km_driven", "engine_cc", "owner_count"]

X = df[FEATURE_COLUMNS]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
    ("num", StandardScaler(), NUMERIC_COLUMNS),
])

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
linear_pipeline = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
linear_pipeline.fit(X_train, y_train)
linear_pred = linear_pipeline.predict(X_test)

print(f"Linear Regression - MAE: {mean_absolute_error(y_test, linear_pred):,.0f}  R2: {r2_score(y_test, linear_pred):.3f}")

In [ ]:
rf_pipeline = Pipeline([("prep", preprocessor), ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

print(f"Random Forest - MAE: {mean_absolute_error(y_test, rf_pred):,.0f}  R2: {r2_score(y_test, rf_pred):.3f}")

In [ ]:
gb_pipeline = Pipeline([("prep", preprocessor), ("model", GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, random_state=42))])
gb_pipeline.fit(X_train, y_train)
gb_pred = gb_pipeline.predict(X_test)

print(f"Gradient Boosting - MAE: {mean_absolute_error(y_test, gb_pred):,.0f}  R2: {r2_score(y_test, gb_pred):.3f}")

## model comparison against baselines

Every model gets scored the same way, alongside two "dumb" baselines - so the comparison is
fair and shows whether the real models are actually earning their complexity.

In [ ]:
baseline_constant_pred = np.full(len(y_test), y_train.mean())
baseline_naive_pred = X_test["year"].map(lambda yr: y_train[X_train["year"] == yr].mean() if (X_train["year"] == yr).any() else y_train.mean()).values

results_table = pd.DataFrame([
    {"Model": "Constant (average price)", "MAE": mean_absolute_error(y_test, baseline_constant_pred), "R2": r2_score(y_test, baseline_constant_pred)},
    {"Model": "Naive (average by year)", "MAE": mean_absolute_error(y_test, baseline_naive_pred), "R2": r2_score(y_test, baseline_naive_pred)},
    {"Model": "Linear Regression", "MAE": mean_absolute_error(y_test, linear_pred), "R2": r2_score(y_test, linear_pred)},
    {"Model": "Random Forest", "MAE": mean_absolute_error(y_test, rf_pred), "R2": r2_score(y_test, rf_pred)},
    {"Model": "Gradient Boosting", "MAE": mean_absolute_error(y_test, gb_pred), "R2": r2_score(y_test, gb_pred)},
]).sort_values("MAE")

results_table

## neural network + zero-shot LLM, compared against trained ML

A small PyTorch network trained on the same data, then an LLM asked to estimate price with
zero training at all - just reasoning from the description. Interesting result to expect:
trained ML models usually beat a zero-shot LLM on this kind of structured numeric task.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)

X_train_tensor = torch.tensor(X_train_processed).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1).to(device)
X_test_tensor = torch.tensor(X_test_processed).to(device)


class PriceNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


nn_model = PriceNet(X_train_processed.shape[1]).to(device)
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Scale target for stable training, unscale predictions afterward
y_mean, y_std = y_train_tensor.mean(), y_train_tensor.std()
y_train_scaled = (y_train_tensor - y_mean) / y_std

for epoch in range(200):
    optimizer.zero_grad()
    pred = nn_model(X_train_tensor)
    loss = loss_fn(pred, y_train_scaled)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: loss {loss.item():.4f}")

nn_model.eval()
with torch.no_grad():
    nn_pred_scaled = nn_model(X_test_tensor)
    nn_pred = (nn_pred_scaled * y_std + y_mean).cpu().numpy().flatten()

print(f"\nNeural Network - MAE: {mean_absolute_error(y_test, nn_pred):,.0f}  R2: {r2_score(y_test, nn_pred):.3f}")

In [ ]:
def llm_price_guess(row, client=openai_client, model=MODEL):
    prompt = (f"Estimate the resale price in INR for this used car. Respond with ONLY a number, "
              f"no currency symbol, no explanation.\n\n"
              f"Brand: {row['brand']}, Year: {row['year']}, KM driven: {row['km_driven']}, "
              f"Fuel: {row['fuel_type']}, Transmission: {row['transmission']}, "
              f"Engine: {row['engine_cc']}cc, Owners: {row['owner_count']}, Location: {row['location']}")
    response = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    try:
        return float(response.choices[0].message.content.strip().replace(",", ""))
    except ValueError:
        return y_train.mean()


# Test on a small sample - LLM calls are slower than a trained model's instant prediction
sample_test = X_test.head(20)
sample_actual = y_test.head(20)
llm_preds = [llm_price_guess(row) for _, row in sample_test.iterrows()]

print(f"Zero-shot LLM (n=20) - MAE: {mean_absolute_error(sample_actual, llm_preds):,.0f}  R2: {r2_score(sample_actual, llm_preds):.3f}")
print(f"Gradient Boosting on same 20 - MAE: {mean_absolute_error(sample_actual, gb_pred[:20]):,.0f}  R2: {r2_score(sample_actual, gb_pred[:20]):.3f}")

## The dashboard

Enter a car's details, get a predicted price, a price range, which factors matter most, and
similar cars from the dataset - using the best-performing model (XGBoost).

In [ ]:
# Gradient Boosting's typical error, used to show a realistic price range rather than a single number
GB_TYPICAL_ERROR = mean_absolute_error(y_test, gb_pred)

feature_names = (gb_pipeline.named_steps["prep"].get_feature_names_out())
importances = gb_pipeline.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False).head(6)


def predict_price(brand, year, km_driven, fuel_type, transmission, engine_cc, owner_count, location):
    input_df = pd.DataFrame([{
        "brand": brand, "year": year, "km_driven": km_driven, "fuel_type": fuel_type,
        "transmission": transmission, "engine_cc": engine_cc, "owner_count": owner_count,
        "location": location
    }])

    predicted = gb_pipeline.predict(input_df)[0]
    low, high = predicted - GB_TYPICAL_ERROR, predicted + GB_TYPICAL_ERROR

    factors_text = "\n".join(f"- {row['feature']}: {row['importance']:.1%}" for _, row in importance_df.iterrows())

    similar = df[(df["brand"] == brand) & (abs(df["year"] - year) <= 2)].sort_values(
        "year", ascending=False).head(5)[["brand", "year", "km_driven", "price"]]
    similar_text = similar.to_string(index=False) if not similar.empty else "No close matches found in the dataset."

    accuracy_text = f"Gradient Boosting model: MAE ₹{GB_TYPICAL_ERROR:,.0f}, R² {r2_score(y_test, gb_pred):.3f} on held-out test data"

    return (f"₹{predicted:,.0f}", f"₹{low:,.0f} - ₹{high:,.0f}", factors_text, similar_text, accuracy_text)


with gr.Blocks(title="Used Car Price Intelligence") as ui:
    gr.Markdown("# 🚗 Used Car Price Intelligence & Prediction System")

    with gr.Row():
        with gr.Column():
            brand_in = gr.Dropdown(list(BRANDS.keys()), value="Honda", label="Brand")
            year_in = gr.Slider(2010, 2024, value=2019, step=1, label="Year")
            km_in = gr.Number(value=45000, label="Kilometers Driven")
            fuel_in = gr.Dropdown(FUEL_TYPES, value="Petrol", label="Fuel Type")
            transmission_in = gr.Dropdown(TRANSMISSIONS, value="Manual", label="Transmission")
            engine_in = gr.Number(value=1500, label="Engine (cc)")
            owner_in = gr.Slider(1, 4, value=1, step=1, label="Number of Previous Owners")
            location_in = gr.Dropdown(LOCATIONS, value="Mumbai", label="Location")
            predict_button = gr.Button("Predict Price", variant="primary")

        with gr.Column():
            price_out = gr.Textbox(label="Estimated Resale Price")
            range_out = gr.Textbox(label="Expected Price Range")
            factors_out = gr.Textbox(label="Most Important Factors", lines=6)
            similar_out = gr.Textbox(label="Similar Cars in Dataset", lines=6)
            accuracy_out = gr.Textbox(label="Model Accuracy")

    predict_button.click(
        predict_price,
        inputs=[brand_in, year_in, km_in, fuel_in, transmission_in, engine_in, owner_in, location_in],
        outputs=[price_out, range_out, factors_out, similar_out, accuracy_out]
    )

ui.launch(share=True)